# BOQ Generator
### AI Estimation Agent - Nagaland PWD SOR 2021

Takes your structure details and calculates all quantities using standard civil engineering formulas.

**Two ways to use:**
- **Option A** - Type your building specs directly (use this now)
- **Option B** - Load from `structure_model.json` (use after structure reader)

Output: `boq.json` and `boq.csv` -> feeds into AI Agent -> Abstract of Cost

**Run every cell top to bottom**

## Step 1 - Install & Imports
*(Run once)*

In [ ]:
!pip install pandas openpyxl groq -q

import json
import math
import pandas as pd
from dataclasses import dataclass, field, asdict
from IPython.display import display
import os
print("Ready")

## Step 2 - BOQ Item Format
Each BOQ item has: description, quantity, unit, and notes.

In [ ]:
@dataclass
class BOQItem:
    sl_no:       int
    category:    str      # Earthwork / Foundation / Superstructure / Finishes
    description: str      # work item description
    quantity:    float
    unit:        str
    notes:       str = '' # assumptions made

print("BOQ Item format defined")

## Step 3 - Quantity Calculator
Calculates quantities using standard PWD civil engineering formulas.

Formulas used:
- **Earthwork** = plinth area x excavation depth (+ 0.3m working space each side)
- **PCC** = footing area x 0.1m thickness
- **Footing RCC** = L x B x D x count
- **Column RCC** = width x depth x height x count x floors
- **Beam RCC** = width x depth x total length
- **Slab RCC** = area x thickness x floors
- **Brickwork** = wall length x height x thickness - openings
- **Plaster** = 2 x (L+B) x H x floors - openings
- **Flooring** = plinth area x floors
- **Steel** = 1% of RCC volume x 7850 kg/cum (standard assumption)

In [ ]:
class BOQCalculator:
    """
    Calculates BOQ quantities from building specifications.
    All standard PWD India formulas.
    """

    def __init__(self, specs):
        self.s = specs
        self.boq = []
        self._sl = 0

    def _add(self, category, description, quantity, unit, notes=''):
        if quantity > 0:
            self._sl += 1
            self.boq.append(BOQItem(
                sl_no       = self._sl,
                category    = category,
                description = description,
                quantity    = round(quantity, 3),
                unit        = unit,
                notes       = notes
            ))

    def calculate(self):
        s = self.s
        print('Calculating quantities...')

        # ── Shorthand variables ──────────────────────────────────────────
        L            = s['length_m']
        B            = s['width_m']
        floors       = s['floors']
        ffl          = s['floor_height_m']        # floor to floor height
        col_w        = s['column_width_m']
        col_d        = s['column_depth_m']
        col_count    = s['columns_per_floor']
        beam_w       = s['beam_width_m']
        beam_d       = s['beam_depth_m']
        slab_t       = s['slab_thickness_m']
        wall_ext_t   = s['external_wall_thickness_m']
        wall_int_t   = s['internal_wall_thickness_m']
        int_wall_len = s['internal_wall_length_m']    # total length of internal walls per floor
        ftg_l        = s['footing_length_m']
        ftg_b        = s['footing_width_m']
        ftg_d        = s['footing_depth_m']
        exc_d        = s['excavation_depth_m']
        doors        = s['doors_per_floor']
        door_w       = s['door_width_m']
        door_h       = s['door_height_m']
        windows      = s['windows_per_floor']
        win_w        = s['window_width_m']
        win_h        = s['window_height_m']
        plinth_area  = L * B
        perimeter    = 2 * (L + B)

        print(f'  Plinth area    : {plinth_area:.2f} sqm')
        print(f'  Perimeter      : {perimeter:.2f} m')
        print(f'  Total columns  : {col_count * floors}')
        print()

        # ════════════════════════════════════════════════════════════════
        # A. EARTHWORK
        # ════════════════════════════════════════════════════════════════
        # Excavation area = plinth + 0.6m extra all sides for working space
        exc_area = (L + 0.6) * (B + 0.6)
        exc_vol  = exc_area * exc_d
        self._add('A. Earthwork',
            'Earth work excavation in foundation trenches, all kinds of soil, '
            'depth upto 1.5m, disposal within 50m lead',
            exc_vol, 'Cum',
            f'({L+0.6:.2f} x {B+0.6:.2f} x {exc_d}m)')

        # Backfilling
        total_rcc_below = col_count * ftg_l * ftg_b * ftg_d
        backfill = exc_vol - total_rcc_below
        self._add('A. Earthwork',
            'Filling in foundation with excavated earth, compacted in layers',
            max(backfill, 0), 'Cum',
            'Excavation vol minus footing concrete vol')

        # ════════════════════════════════════════════════════════════════
        # B. FOUNDATION
        # ════════════════════════════════════════════════════════════════
        # PCC under footings
        pcc_area = col_count * (ftg_l + 0.1) * (ftg_b + 0.1)  # 50mm extra each side
        self._add('B. Foundation',
            'Providing and laying PCC 1:4:8 (M7.5) in foundation, '
            '100mm thick, including formwork',
            pcc_area * 0.1, 'Cum',
            f'{col_count} footings x ({ftg_l+0.1:.2f} x {ftg_b+0.1:.2f} x 0.1m)')

        # Footing RCC
        ftg_vol = col_count * ftg_l * ftg_b * ftg_d
        self._add('B. Foundation',
            'Providing and laying RCC M20 in isolated footings, '
            'including formwork, curing complete',
            ftg_vol, 'Cum',
            f'{col_count} footings x ({ftg_l} x {ftg_b} x {ftg_d}m)')

        # ════════════════════════════════════════════════════════════════
        # C. SUPERSTRUCTURE - RCC
        # ════════════════════════════════════════════════════════════════
        # Columns
        col_vol = col_count * floors * col_w * col_d * ffl
        self._add('C. Superstructure - RCC',
            'Providing and laying RCC M20 in columns, '
            'including shuttering, centering, curing complete',
            col_vol, 'Cum',
            f'{col_count} cols x {floors} floors x ({col_w} x {col_d} x {ffl}m)')

        # Beams - total beam length per floor = perimeter + internal grid
        # Approximate internal beams = (col grid spans)
        beam_len_per_floor = perimeter + int_wall_len
        beam_vol = beam_len_per_floor * beam_w * beam_d * floors
        self._add('C. Superstructure - RCC',
            'Providing and laying RCC M20 in beams, '
            'including shuttering, centering, curing complete',
            beam_vol, 'Cum',
            f'{beam_len_per_floor:.1f}m beam length x ({beam_w} x {beam_d}m) x {floors} floors')

        # Slabs
        slab_vol = plinth_area * slab_t * floors
        self._add('C. Superstructure - RCC',
            'Providing and laying RCC M20 in slabs, '
            'including shuttering, centering, curing complete',
            slab_vol, 'Cum',
            f'{plinth_area:.2f} sqm x {slab_t}m thick x {floors} floors')

        # Staircase RCC (approximate)
        if s.get('staircases', 0) > 0:
            stair_vol = s['staircases'] * 3.0 * (floors - 1)  # ~3 cum per flight
            self._add('C. Superstructure - RCC',
                'Providing and laying RCC M20 in staircases, '
                'including shuttering, centering, curing complete',
                stair_vol, 'Cum',
                f'{s["staircases"]} staircase x {floors-1} flights x 3.0 cum/flight (approx)')

        # Steel reinforcement (1% of total RCC volume is standard assumption)
        total_rcc = col_vol + beam_vol + slab_vol + ftg_vol
        steel_kg  = total_rcc * 0.01 * 7850  # 1% steel, density 7850 kg/cum
        self._add('C. Superstructure - RCC',
            'Providing, cutting, bending and placing TMT steel '
            'reinforcement Fe415/Fe500, including binding wire',
            steel_kg, 'Kg',
            f'1% of total RCC {total_rcc:.2f} cum x 7850 kg/cum')

        # ════════════════════════════════════════════════════════════════
        # D. BRICKWORK
        # ════════════════════════════════════════════════════════════════
        wall_height = ffl - beam_d  # wall height = floor height minus beam

        # External walls - deduct columns from wall length
        ext_wall_len = perimeter - (col_count ** 0.5 * 4 * col_w)  # approx column deduction
        ext_wall_area = ext_wall_len * wall_height * floors
        # Deduct openings
        opening_area = (doors * door_w * door_h + windows * win_w * win_h) * floors
        ext_wall_vol = (ext_wall_area - opening_area * 0.5) * wall_ext_t  # 50% openings on ext walls
        self._add('D. Brickwork',
            'Brick masonry in superstructure, in CM 1:6, '
            'with modular bricks, including scaffolding',
            max(ext_wall_vol, 0), 'Cum',
            f'External: {ext_wall_len:.1f}m x {wall_height:.2f}m high x {wall_ext_t}m thick x {floors} floors, minus openings')

        # Internal partition walls
        int_wall_area = int_wall_len * wall_height * floors
        int_opening   = (doors * door_w * door_h) * floors * 0.5  # approx
        int_wall_vol  = (int_wall_area - int_opening) * wall_int_t
        self._add('D. Brickwork',
            'Brick masonry in partition walls, in CM 1:4, '
            'half brick thick (115mm), including scaffolding',
            max(int_wall_vol, 0), 'Cum',
            f'Internal: {int_wall_len:.1f}m x {wall_height:.2f}m high x {wall_int_t}m thick x {floors} floors')

        # ════════════════════════════════════════════════════════════════
        # E. FINISHES
        # ════════════════════════════════════════════════════════════════
        # Internal plaster
        int_plaster_area = (
            (ext_wall_area + int_wall_area * 2) - opening_area
        )
        self._add('E. Finishes',
            'Cement plaster 12mm thick in CM 1:4, '
            'internal walls and ceiling, finished smooth',
            int_plaster_area, 'Sqm',
            f'Walls + ceiling area minus openings')

        # External plaster
        ext_plaster_area = ext_wall_area - (opening_area * 0.5)
        self._add('E. Finishes',
            'Cement plaster 15mm thick in CM 1:5, '
            'external walls, finished smooth',
            max(ext_plaster_area, 0), 'Sqm',
            f'External wall area minus openings')

        # Flooring
        floor_area = plinth_area * floors
        self._add('E. Finishes',
            'Providing and laying vitrified tiles 600x600mm, '
            'in CM 1:3, including grouting',
            floor_area, 'Sqm',
            f'{plinth_area:.2f} sqm x {floors} floors')

        # Skirting (10% of floor area as linear measure approx)
        self._add('E. Finishes',
            'Providing and laying skirting tiles, '
            '100mm height, in CM 1:3',
            perimeter * floors * 0.1, 'Sqm',
            f'Perimeter {perimeter:.1f}m x {floors} floors x 0.1m height')

        # White wash / paint (ceiling)
        ceiling_area = plinth_area * floors
        self._add('E. Finishes',
            'Two coats of white wash over plastered surface, ceiling',
            ceiling_area, 'Sqm',
            f'{plinth_area:.2f} sqm x {floors} floors')

        # ════════════════════════════════════════════════════════════════
        # F. DOORS & WINDOWS
        # ════════════════════════════════════════════════════════════════
        total_doors = doors * floors
        self._add('F. Doors & Windows',
            f'Providing and fixing panelled teak wood door frame '
            f'and shutters {int(door_w*1000)}x{int(door_h*1000)}mm',
            total_doors, 'Nos',
            f'{doors} per floor x {floors} floors')

        total_windows = windows * floors
        self._add('F. Doors & Windows',
            f'Providing and fixing aluminium glazed windows '
            f'{int(win_w*1000)}x{int(win_h*1000)}mm, with glass',
            total_windows, 'Nos',
            f'{windows} per floor x {floors} floors')

        print(f'  Total BOQ items generated: {len(self.boq)}')
        return self.boq


print("BOQ Calculator defined")

## Step 4 - Enter Your Building Specs

**Option A - Type your specs directly** (use this now)

Edit the values below to match your actual building.
All dimensions in **metres**.

In [ ]:
# ── EDIT YOUR BUILDING SPECS HERE ──────────────────────────
specs = {
    # Overall building
    'length_m':                  10.0,   # building length
    'width_m':                    8.0,   # building width
    'floors':                       3,   # number of floors (G+2 = 3)
    'floor_height_m':             3.2,   # floor to floor height

    # Columns
    'column_width_m':            0.23,   # column width
    'column_depth_m':            0.45,   # column depth
    'columns_per_floor':           12,   # number of columns per floor

    # Beams
    'beam_width_m':              0.23,   # beam width
    'beam_depth_m':              0.40,   # beam depth (excluding slab)

    # Slab
    'slab_thickness_m':         0.125,   # slab thickness

    # Walls
    'external_wall_thickness_m': 0.23,   # external wall thickness
    'internal_wall_thickness_m': 0.115,  # internal partition wall
    'internal_wall_length_m':    25.0,   # total internal wall length per floor

    # Foundation
    'footing_length_m':           1.8,   # footing length
    'footing_width_m':            1.8,   # footing width
    'footing_depth_m':           0.45,   # footing depth
    'excavation_depth_m':         1.5,   # excavation depth

    # Openings per floor
    'doors_per_floor':              8,
    'door_width_m':               0.9,
    'door_height_m':              2.1,
    'windows_per_floor':           12,
    'window_width_m':             1.2,
    'window_height_m':            1.2,

    # Staircases
    'staircases':                   1,
}
# ────────────────────────────────────────────────────────────

plinth = specs['length_m'] * specs['width_m']
print(f"Building    : {specs['length_m']}m x {specs['width_m']}m")
print(f"Plinth area : {plinth:.2f} sqm  ({plinth*10.764:.0f} sqft)")
print(f"Floors      : G+{specs['floors']-1} ({specs['floors']} floors total)")
print(f"Total height: {specs['floors'] * specs['floor_height_m']:.1f} m")

**Option B - Load from structure_model.json** (use after running structure reader)

Only run this cell if you have already run `1_structure_reader.ipynb`.

In [ ]:
# Only run this if structure_model.json exists
if os.path.exists('structure_model.json'):
    with open('structure_model.json') as f:
        sm = json.load(f)

    # Auto-fill specs from structure model
    # Pull first column/beam/slab member values
    def _first(lst, key, default):
        return lst[0].get(key, default) if lst else default

    cols = sm.get('columns', [])
    beams = sm.get('beams', [])
    slabs = sm.get('slabs', [])
    walls = sm.get('walls', [])
    ftgs  = sm.get('footings', [])

    import math
    plinth = sm.get('plinth_area_sqm', 80)
    side   = math.sqrt(plinth)

    specs = {
        'length_m':                  round(side * 1.25, 1),
        'width_m':                   round(side * 0.8, 1),
        'floors':                    sm.get('floors', 1),
        'floor_height_m':            sm.get('total_height_m', 3.2) / max(sm.get('floors',1), 1),
        'column_width_m':            _first(cols, 'width_m', 0.23),
        'column_depth_m':            _first(cols, 'depth_m', 0.45),
        'columns_per_floor':         _first(cols, 'count', 12),
        'beam_width_m':              _first(beams, 'width_m', 0.23),
        'beam_depth_m':              _first(beams, 'depth_m', 0.40),
        'slab_thickness_m':          _first(slabs, 'depth_m', 0.125),
        'external_wall_thickness_m': 0.23,
        'internal_wall_thickness_m': 0.115,
        'internal_wall_length_m':    round(plinth / 3, 1),
        'footing_length_m':          _first(ftgs, 'length_m', 1.8),
        'footing_width_m':           _first(ftgs, 'width_m', 1.8),
        'footing_depth_m':           _first(ftgs, 'depth_m', 0.45),
        'excavation_depth_m':        1.5,
        'doors_per_floor':           len(sm.get('doors', [{'count':8}])) and sm.get('doors',[{}])[0].get('count',8),
        'door_width_m':              0.9,
        'door_height_m':             2.1,
        'windows_per_floor':         sm.get('windows',[{}])[0].get('count',12) if sm.get('windows') else 12,
        'window_width_m':            1.2,
        'window_height_m':           1.2,
        'staircases':                len(sm.get('staircases',[])),
    }
    print('Specs loaded from structure_model.json')
    print(f"Plinth area : {plinth} sqm")
    print(f"Floors      : {specs['floors']}")
else:
    print('structure_model.json not found - use Option A above')

## Step 5 - Calculate BOQ
Runs all the formulas on your specs.

In [ ]:
print('=' * 55)
print('  BOQ CALCULATION')
print('=' * 55)

calc = BOQCalculator(specs)
boq_items = calc.calculate()

# Display as table
rows = []
for item in boq_items:
    rows.append({
        'Sl.No':      item.sl_no,
        'Category':   item.category,
        'Description': item.description[:70] + '...' if len(item.description) > 70 else item.description,
        'Quantity':   item.quantity,
        'Unit':       item.unit,
    })

df_boq = pd.DataFrame(rows)
print()
display(df_boq)

## Step 6 - Review by Category
Check quantities category by category.

In [ ]:
print('=' * 55)
print('  BOQ SUMMARY BY CATEGORY')
print('=' * 55)

for cat in df_boq['Category'].unique():
    items = df_boq[df_boq['Category'] == cat]
    print(f'\n  {cat}')
    print('  ' + '-'*50)
    for _, row in items.iterrows():
        print(f"  {row['Sl.No']:2d}. {row['Description'][:55]:55s}")
        print(f"      {row['Quantity']:>10.3f} {row['Unit']}")

## Step 7 - Save BOQ
Saves `boq.json` and `boq.csv` for the AI Agent.

In [ ]:
# Save to JSON
boq_data = [asdict(item) for item in boq_items]
with open('boq.json', 'w') as f:
    json.dump(boq_data, f, indent=2)

# Save to CSV
df_full = pd.DataFrame(boq_data)
df_full.to_csv('boq.csv', index=False)

print(f'Saved: boq.json  ({len(boq_items)} items)')
print(f'Saved: boq.csv   ({len(boq_items)} items)')
print()
print('=' * 55)
print('NEXT: Open AI_agent.ipynb')
print('It reads boq.json + sor_vectordb and produces')
print('the final Abstract of Cost Excel file.')
print('=' * 55)